In [1]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMModel
from algorithm import poc1

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

ImportError: cannot import name 'poc1' from 'algorithm' (C:\Users\User\Desktop\newtests\algorithm.py)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')
else:
    device = torch.device("cpu")

# Parameters

In [ ]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 50
n_layers = 2
lr = 0.0001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 5 #number of clients to participate in training

# Data Preperation

In [ ]:
dataframes = fetch_dataset("./cleaned_data18")

In [ ]:
n_clients = len(dataframes) #total number of clients to partition data into

total_data_points = sum(df.shape[0] for df in dataframes.values()) # Compute the total number of data points

# Compute the data ratios
data_ratios = [df.shape[0] / total_data_points for client_id, df in dataframes.items()]

In [ ]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

# Build the training set

In [ ]:
train_loader = []
test_loader = []
label_scalers = []
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to('cpu'), torch.from_numpy(train_y).to('cpu'))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to('cpu'), torch.from_numpy(test_y).to('cpu'))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y
input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

# LSTM model

In [ ]:
#         self.global_parameter = torch.zeros(self.num_classes, self.dim)
        
#         # defining the model: here, logistic regression
#         self.model = torch.nn.Linear(self.dim, self.num_classes, bias=False)
        
#         # defining loss function: here, cross-entropy
#         self.criterion = torch.nn.CrossEntropyLoss()

#         # defining the optimizer: here, SGD
#         self.optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr, weight_decay=10e-4)

In [ ]:
lstm = LSTMModel(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

## Power Of Choice

In [ ]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc1(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd=8,
    num_clients_per_round=5,
    ratio = data_ratios
)

In [11]:
# np.save(f'metrics/fedavg_lstm_outputs_C{num_clients_per_round}-.npy', np.array(outputs_avg, dtype=object))
# np.save(f'metrics/fedavg_lstm_targets_C{num_clients_per_round}-.npy', np.array(targets_avg, dtype=object))
# np.save(f'metrics/fedavg_lstm_loss_C{num_clients_per_round}-.npy', loss_avg)
# np.save(f'metrics/fedavg_lstm_smape_C{num_clients_per_round}-.npy', smape_avg)
# np.save(f'metrics/fedavg_lstm_mae_C{num_clients_per_round}-.npy', mae_avg)
# np.save(f'metrics/fedavg_lstm_rmse_C{num_clients_per_round}-.npy', rmse_avg)